# NBA Player Season Data — ETL

Fetches all player-season stats from the NBA API and saves `player_seasons.json`.
Run all cells top to bottom. The last cell pushes the result directly to GitHub.

**You'll need a GitHub personal access token** with `repo` (Contents: read & write) scope.
Create one at: GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q nba_api pandas requests

In [ ]:
# ── 2. Config — fill these in ─────────────────────────────────────────────────

GITHUB_TOKEN = ""        # GitHub PAT with repo write access
GITHUB_OWNER = "JoesCrepes"
GITHUB_REPO  = "nba-curios"
GITHUB_BRANCH = "main"
GITHUB_FILE_PATH = "public/data/player_seasons.json"

# Season range to fetch (inclusive)
SEASON_START = "1996-97"
SEASON_END   = "2024-25"

# Set to True to only re-fetch the current season (fast, ~30s)
# Set to False for a full historical seed (~20 min)
UPDATE_CURRENT_ONLY = False

In [ ]:
# ── 3. Connectivity test ──────────────────────────────────────────────────────
# Run this first to confirm stats.nba.com is reachable from this environment.

import requests, time

NBA_HEADERS = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'x-nba-stats-origin': 'stats',
    'x-nba-stats-token': 'true',
    'Origin': 'https://www.nba.com',
    'Referer': 'https://www.nba.com/',
    'Sec-Fetch-Site': 'same-site',
    'Sec-Fetch-Mode': 'cors',
    'Connection': 'keep-alive',
}

test_url = (
    'https://stats.nba.com/stats/leaguedashplayerstats'
    '?Season=2023-24&SeasonType=Regular+Season&MeasureType=Base'
    '&PerMode=Totals&LastNGames=0&Month=0&OpponentTeamID=0'
    '&PaceAdjust=N&PlusMinus=N&Rank=N&LeagueID=00'
    '&PlayerPosition=&TeamID=&College=&Conference=&Country='
    '&DateFrom=&DateTo=&Division=&DraftPick=&DraftYear='
    '&GameScope=&GameSegment=&Height=&Location=&Outcome='
    '&PORound=&Period=0&PlayerExperience=&SeasonSegment='
    '&ShotClockRange=&StarterBench=&TwoWay=&VsConference=&VsDivision=&Weight='
)

try:
    r = requests.get(test_url, headers=NBA_HEADERS, timeout=60)
    data = r.json()
    n = len(data['resultSets'][0]['rowSet'])
    print(f'✓ Connected — got {n} players for 2023-24')
except Exception as e:
    print(f'✗ Failed: {e}')
    print('stats.nba.com is not reachable from this environment.')

In [ ]:
# ── 4. ETL ────────────────────────────────────────────────────────────────────

import json
import time
from datetime import datetime
import pandas as pd
from nba_api.stats.endpoints import leaguedashplayerstats

API_TIMEOUT   = 60
MAX_RETRIES   = 3
RETRY_BACKOFF = [2, 4, 8]

def season_range(start, end):
    s, e = int(start[:4]), int(end[:4])
    return [f"{y}-{str(y+1)[-2:]}" for y in range(s, e+1)]

def fetch_season(season):
    print(f'  {season}...', end='', flush=True)
    time.sleep(0.7)
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star='Regular Season',
                measure_type_detailed_defense='Base',
                per_mode_detailed='Totals',
                timeout=API_TIMEOUT,
                headers=NBA_HEADERS,
            )
            df = resp.get_data_frames()[0]
            df['SEASON'] = season
            print(f' {len(df)} players')
            return df
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES - 1:
                wait = RETRY_BACKOFF[attempt]
                print(f' timeout, retrying in {wait}s...', end='', flush=True)
                time.sleep(wait)
    raise RuntimeError(f'Failed after {MAX_RETRIES} attempts: {last_err}')

def process_season(df, season):
    rows = []
    has_pos = 'PLAYER_POSITION' in df.columns
    for _, r in df.iterrows():
        fgm  = int(r.get('FGM', 0))
        fga  = int(r.get('FGA', 0))
        fg3m = int(r.get('FG3M', 0))
        fg3a = int(r.get('FG3A', 0))
        gp   = int(r.get('GP', 1)) or 1
        two_pm  = fgm - fg3m
        two_pa  = fga - fg3a
        two_pct = (two_pm / two_pa) if two_pa > 0 else 0.0
        rows.append({
            'id': int(r['PLAYER_ID']), 'name': str(r['PLAYER_NAME']),
            'season': season, 'team': str(r.get('TEAM_ABBREVIATION', '')),
            'pos': str(r.get('PLAYER_POSITION', '')) if has_pos else '',
            'gp': gp,
            'fgm': fgm, 'fga': fga, 'fg_pct': round(float(r.get('FG_PCT', 0) or 0), 4),
            'fg3m': fg3m, 'fg3a': fg3a, 'fg3_pct': round(float(r.get('FG3_PCT', 0) or 0), 4),
            'ftm': int(r.get('FTM', 0)), 'fta': int(r.get('FTA', 0)),
            'ft_pct': round(float(r.get('FT_PCT', 0) or 0), 4),
            'two_pm': two_pm, 'two_pa': two_pa, 'two_pct': round(two_pct, 4),
            'oreb': int(r.get('OREB', 0)), 'dreb': int(r.get('DREB', 0)),
            'reb': int(r.get('REB', 0)), 'ast': int(r.get('AST', 0)),
            'stl': int(r.get('STL', 0)), 'blk': int(r.get('BLK', 0)),
            'pts': int(r.get('PTS', 0)), 'tov': int(r.get('TOV', 0)),
        })
    return rows

# ── Run ───────────────────────────────────────────────────────────────────────

OUTPUT_FILE = 'player_seasons.json'
existing_rows = []

if UPDATE_CURRENT_ONLY:
    try:
        with open(OUTPUT_FILE) as f:
            existing = json.load(f)
        existing_rows = [r for r in existing.get('data', []) if r['season'] != SEASON_END]
        print(f'Loaded {len(existing_rows)} existing rows (excluding {SEASON_END})')
    except FileNotFoundError:
        pass
    seasons = [SEASON_END]
else:
    seasons = season_range(SEASON_START, SEASON_END)

print(f'\nFetching {len(seasons)} season(s)...\n')

all_rows = []
errors   = []
for season in seasons:
    try:
        df   = fetch_season(season)
        rows = process_season(df, season)
        all_rows.extend(rows)
    except Exception as e:
        print(f'  ERROR: {e}')
        errors.append(season)

final_rows = existing_rows + all_rows
final_rows.sort(key=lambda r: (r['season'], r['name']))
seasons_covered = sorted(set(r['season'] for r in final_rows))

output = {
    'meta': {
        'fetched_at': datetime.now().isoformat(),
        'seasons_covered': seasons_covered,
        'total_rows': len(final_rows),
    },
    'data': final_rows,
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, separators=(',', ':'))

import os
size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f'\n✓ Saved {len(final_rows):,} rows ({size_kb:.0f} KB)')
if seasons_covered:
    print(f'  Seasons: {seasons_covered[0]} → {seasons_covered[-1]}')
if errors:
    print(f'  Failed seasons: {errors}')

In [ ]:
# ── 5. Push to GitHub ─────────────────────────────────────────────────────────
# Pushes player_seasons.json directly to the repo via the GitHub Contents API.
# Requires GITHUB_TOKEN set in the Config cell above.

import base64, requests as req

if not GITHUB_TOKEN:
    print('Set GITHUB_TOKEN in the Config cell first.')
else:
    api_url = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{GITHUB_FILE_PATH}'
    gh_headers = {
        'Authorization': f'Bearer {GITHUB_TOKEN}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }

    # Get current file SHA (required by GitHub API to update an existing file)
    existing = req.get(f'{api_url}?ref={GITHUB_BRANCH}', headers=gh_headers)
    sha = existing.json().get('sha') if existing.status_code == 200 else None

    with open(OUTPUT_FILE, 'rb') as f:
        encoded = base64.b64encode(f.read()).decode()

    payload = {
        'message': f'Seed player season data via Colab ({datetime.now().strftime("%Y-%m-%d")})',
        'content': encoded,
        'branch': GITHUB_BRANCH,
    }
    if sha:
        payload['sha'] = sha

    r = req.put(api_url, headers=gh_headers, json=payload)
    if r.status_code in (200, 201):
        commit_url = r.json()['commit']['html_url']
        print(f'✓ Pushed to {GITHUB_BRANCH}: {commit_url}')
    else:
        print(f'✗ GitHub API error {r.status_code}: {r.json().get("message")}')